# 11.a — Ad hoc test: risk indicators on today's cropland (EC-Earth3 runs)

**Ad hoc test, outside the suitability chain**: it reads the masks of 0.1, the calendars of the ERA5 BIO files of
1.1.d and the daily climate of the configuration, and writes only in `<out_root>/test_risk_indicators/<route>/`.

**Question.** The suitability (SVM) works on 30-year means: a cell can stay suitable on average and see many more bad
seasons. Where the crops are grown today, how often do the bad seasons come, and does the AMOC collapse make them more
or less frequent? Approach of Ceglar et al. (2021, *Environ. Res. Lett.* 16, 104049, sections 2.4 and 3.1, figures 1–2),
on the crops and calendars of this chain.

**Domain.** The cells where the crop is grown today (MIRCA2000 presence of the route, as the labels of 1.1.d), weighted
by the harvested area; the calendar of the ERA5 BIO file (sowing, anthesis = start of phase 2 of variant A, maturity),
fixed in every scenario.

**Indicators, per crop, cell and season.**
- **Heat**: number of days with Tmax above the critical temperature of the crop between anthesis and maturity (maize
  35 °C, wheat 34 °C, rice 36 °C, soybean 39 °C, Gourdji et al. 2013, *Environ. Res. Lett.* 8, 024041; potato 30 °C,
  provisional). Ceglar et al. count the days above 28 °C for durum wheat.
- **Drought**: SPEI (Vicente-Serrano et al. 2010, *J. Climate* 23, 1696–1718) of the season (sowing → maturity) and of
  phase 2: the water balance Σ (pr − ET0) of the window (ET0 Hargreaves, as 1.1.d), a log-logistic distribution
  fitted per cell on the reference seasons by L-moments (section 2), then the standard normal quantile.

**Events, relative to the reference of each cell** (EC-Earth3 historical 1981–2014, raw): a **hot season** has more hot
days than the 75th percentile of the reference seasons (Ceglar et al.); a **drought** has SPEI ≤ −1.5 (severe) or ≤ −2
(extreme). By construction the reference has about 25 % hot seasons (fewer where its 75th percentile is 0) and 6.7 % /
2.3 % droughts. The thresholds are relative to the climate of the model, so a constant bias of EC-Earth3 cancels: the
runs are used **raw**, with their own sequences (persistence, year-to-year variability). Caveat: the hot-day count uses
an absolute threshold, so where EC-Earth3 is too cold or too hot the count itself is biased; section 5 compares it with
ERA5.

**11.b: threshold mapped to EC-Earth3 (`RISK_HEAT=mapped`).** EC-Earth3 is too hot at the crop thresholds on the
cropland (first run: hot days per season maize 3.5 vs 1.4 in ERA5, rice 1.9 vs 0.7, soybean 0.6 vs 0.05), so the
absolute Tcrit counts part of its bias. Instead of correcting the data, the threshold is corrected, cell by cell: the
quantile q of Tcrit in the daily Tmax of phase 2 of ERA5 (1981–2014, all seasons) gives the threshold of EC-Earth3 = its
own quantile q of the same days of EC-Earth3 historical (quantile mapping of the threshold; the runs stay raw, with
their own sequences). Beyond the 99th (or below the 1st) percentile of ERA5, where Tcrit is rarely or never reached,
the threshold is shifted by the difference of that percentile between EC-Earth3 and ERA5. The drought (SPEI) is
unchanged. Outputs in `heat_mapped/`; section 5 then compares the hot days of ERA5 at Tcrit with those of EC-Earth3
historical at its mapped threshold (equal if the mapping works).

**Outputs.** Per crop and scenario: frequency of the events per cell (maps), area-weighted over today's cropland
(globally and in the 6 AR6 land regions with the largest area of the crop; Iturbide et al. 2020), probability of
exceedance of the share of the cropland affected in a season (figure 2 of Ceglar et al.), hosing − no hosing.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import time
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import norm
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu
CFG = cu.load_config()

# ---- parameters
CHOSEN_VARIANT = "A_60d"          # calendar of the ERA5 BIO files (anthesis = start of phase 2)
CELL_AREA_THR = 0.0005
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
ERA5_NAME, ERA5_PERIOD = "ERA5", ("1981", "2014")    # BIO files of 1.1.d (calendar)
REF, REF_PERIOD = "gcm_historical", ("1981", "2014")  # reference of the events: EC-Earth3 historical, raw
RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]   # whole files, raw
RUN_PERIOD = {}                   # run -> (first, last year) to read only part of a file; whole file by default
PAIRS = [("gcm_2025_hosing", "gcm_2025_nohosing"), ("gcm_2050_hosing", "gcm_2050_nohosing")]   # hosing - original
HEAT_TCRIT = {"Maize": 35.0, "Wheat": 34.0, "Rice": 36.0, "Soybean": 39.0, "Potato": 30.0}   # degC, as 10.a
HEAT_PCT = 75                     # hot season: more hot days than this percentile of the reference seasons
SPEI_THR = (-1.5, -2.0)           # drought: SPEI <= threshold (severe, extreme)
N_REGIONS = 6                     # AR6 land regions with the largest area of the crop
ERA5_CHECK = True                 # section 5: hot days of ERA5 vs EC-Earth3 historical (bias of the count)
RISK_HEAT = os.environ.get("RISK_HEAT", "absolute")   # "absolute": Tcrit everywhere (11.a) | "mapped": Tcrit mapped
                                  # cell by cell to its EC-Earth3 quantile (11.b); set for one run with the environment
TAIL_Q = 0.01                     # mapped: beyond the 1st / 99th percentile of ERA5, shift by the difference of that percentile
BIN = (-60.0, 70.0, 0.1)          # degC: bins of the Tmax histograms of the mapping
assert RISK_HEAT in ("absolute", "mapped"), RISK_HEAT

ROUTE = cu.route(CELL_AREA_THR)
TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_{ROUTE}", CHOSEN_VARIANT)
OUT = cu.out_dir(CFG, "test_risk_indicators", ROUTE, *(["heat_mapped"] if RISK_HEAT == "mapped" else []))   # the only folder written
SCEN = ["historical", *RUNS]
SPEC = {"historical": (REF, REF_PERIOD), **{r: (r, RUN_PERIOD.get(r)) for r in RUNS}}

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
base_of = lambda crop: crop.split("_")[0].split("-")[0]

# today's cropland of each crop (presence of the route, as the labels of 1.1.d), harvested area, calendar
CROP = {}
for crop in CROPS:
    cm = cu.crop_mask(CFG, crop)
    with xr.open_dataset(ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc") as ds:
        cal = {v: ds[v].values for v in ("sowing_doy", "phase2_start_doy", "maturity_doy")}
    pres = LAND & (cm["presence"].values == 1) & np.isfinite(cal["sowing_doy"]) & np.isfinite(cal["maturity_doy"])
    if CELL_AREA_THR:
        pres &= cu.cell_area_frac(CFG, cm["area_ha"].values) > CELL_AREA_THR
    CROP[crop] = {"mask": pres, "area": np.nan_to_num(cm["area_ha"].values), "cal": cal}
UNION = np.logical_or.reduce([CROP[c]["mask"] for c in CROPS])
UI, UJ = np.nonzero(UNION)
COL = -np.ones(LAND.shape, int); COL[UI, UJ] = np.arange(UI.size)   # column of each cropland cell in the daily arrays
for crop in CROPS:
    i, j = np.nonzero(CROP[crop]["mask"])
    CROP[crop].update({"ij": (i, j), "col": COL[i, j], "w": CROP[crop]["area"][i, j],
                       **{k: CROP[crop]["cal"][k][i, j].astype(float) for k in CROP[crop]["cal"]}})

# AR6 land regions (Iturbide et al. 2020) of the cells
try:
    import regionmask
    AR6 = regionmask.defined_regions.ar6.land
    REG = AR6.mask(GRID["lon"], GRID["lat"]).values                  # (lat, lon), region number or NaN
    REG_NAME = dict(zip(AR6.numbers, AR6.abbrevs))
except Exception as e:                       # no regions: global results only
    print("regionmask not available:", e)
    REG, REG_NAME = np.full(LAND.shape, np.nan), {}
print(f"{len(CROPS)} crops, {UI.size} cropland cells | reference {REF} {REF_PERIOD} | runs {RUNS} | heat threshold {RISK_HEAT} | written only in {OUT}")

## 1. Seasonal indicators of every scenario (daily climate read once per scenario, cropland cells only)

In [ ]:
def compute_Ra(lat_rad, doy):
    Gsc = 0.0820
    dr = 1 + 0.033 * np.cos(2 * np.pi / 365 * doy)
    delta = 0.409 * np.sin(2 * np.pi / 365 * doy - 1.39)
    ws = np.arccos(np.clip(-np.tan(lat_rad) * np.tan(delta), -1, 1))
    return (24 * 60 / np.pi) * Gsc * dr * (ws * np.sin(lat_rad) * np.sin(delta) + np.cos(lat_rad) * np.cos(delta) * np.sin(ws))


def eto_hargreaves(tx, tn, lat_rad, doy):
    # as compute_ETo_day of 1.1.d: Ra in mm/day equivalent (divided by the latent heat of vaporisation)
    tm = (tx + tn) / 2.0
    return 0.0023 * compute_Ra(lat_rad, doy) / (2.501 - 0.002361 * tm) * (tm + 17.8) * np.sqrt(np.maximum(tx - tn, 0.0))


def ylen(y):
    return 366 if (y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)) else 365


def windows(dates, c):
    # the seasons of the crop (harvest year hy, with hy - 1 in the data: the seasons crossing New Year start in hy - 1):
    # day indices and masks (days, cells) of the season and of phase 2
    yr, doy = dates.year.values, dates.dayofyear.values
    years = sorted(set(yr))
    sow, ant, mat = c["sowing_doy"], c["phase2_start_doy"], c["maturity_doy"]
    wrap = mat > 365
    for hy in years:
        if hy - 1 not in years:
            continue
        idx = np.where((yr == hy) | (yr == hy - 1))[0]
        T = (doy[idx] + np.where(yr[idx] == hy, ylen(hy - 1), 0))[:, None]
        shift = np.where(wrap, 0, ylen(hy - 1))[None]
        s, a, m = sow[None] + shift, ant[None] + shift, mat[None] + shift
        yield hy, idx, (T >= s) & (T <= m), (T >= a) & (T <= m)


def seasons(TX, PR, ET, dates, c, tcrit):
    # per season and cell of the crop: hot days (Tmax > tcrit, a number or one value per cell) between anthesis and
    # maturity, water balance of the season and of phase 2
    col = c["col"]
    hys, out = [], {"HOT": [], "D_SEASON": [], "D_P2": []}
    for hy, idx, season, p2 in windows(dates, c):
        wb = PR[idx][:, col] - ET[idx][:, col]
        out["HOT"].append(((TX[idx][:, col] > tcrit) & p2).sum(0))
        out["D_SEASON"].append(np.where(season, wb, 0.0).sum(0))
        out["D_P2"].append(np.where(p2, wb, 0.0).sum(0))
        hys.append(hy)
    return np.array(hys), {k: np.array(v, float) for k, v in out.items()}


NB = int(round((BIN[1] - BIN[0]) / BIN[2]))


def tx_counts(TX, dates, c):
    # histogram (cells, bins) of the daily Tmax of phase 2 over all the seasons
    n, col = c["col"].size, c["col"]
    cnt = np.zeros(n * NB, np.int64)
    for _, idx, _, p2 in windows(dates, c):
        b = np.clip(((TX[idx][:, col] - BIN[0]) / BIN[2]).astype(int), 0, NB - 1)
        cnt += np.bincount((np.arange(n)[None] * NB + b)[p2], minlength=n * NB)
    return cnt.reshape(n, NB)


def cdf(cnt):
    return np.cumsum(cnt, 1) / np.maximum(cnt.sum(1, keepdims=True), 1)


def value_at(cnt, q):
    # value at the quantile q (per cell): upper edge of the first bin where the empirical CDF reaches q
    return BIN[0] + (np.argmax(cdf(cnt) >= np.asarray(q)[:, None] - 1e-12, axis=1) + 1) * BIN[2]


def mapped_threshold(cnt_e, cnt_m, tcrit):
    # threshold of EC-Earth3 = its quantile q of the phase-2 Tmax, q = quantile of tcrit in ERA5; beyond the
    # 1st / 99th percentile of ERA5: tcrit shifted by the difference of that percentile (EC-Earth3 - ERA5)
    n = cnt_e.shape[0]
    q = cdf(cnt_e)[np.arange(n), int(np.clip((tcrit - BIN[0]) / BIN[2], 0, NB - 1))]
    thr = value_at(cnt_m, np.clip(q, TAIL_Q, 1 - TAIL_Q))
    for tail, sel in ((1 - TAIL_Q, q > 1 - TAIL_Q), (TAIL_Q, q < TAIL_Q)):
        qq = np.full(n, tail)
        thr = np.where(sel, tcrit + value_at(cnt_m, qq) - value_at(cnt_e, qq), thr)
    return thr, q


def read_daily(name, period, vars_=("tasmax", "tasmin", "pr")):
    t0 = time.time()
    arr = {}
    for v in vars_:
        da = cu.read_climate(CFG, name, v, GRID, period)
        arr[v] = cu.land_series(da, (UI, UJ))
    dates = pd.DatetimeIndex(da["time"].values).normalize()
    print(f"  {name} {period or 'whole file'}: {dates[0].date()} -> {dates[-1].date()} ({(time.time() - t0) / 60:.1f} min)", flush=True)
    return arr, dates


LATR = np.radians(GRID["lat"][UI])
TC = {c: np.full(CROP[c]["col"].size, HEAT_TCRIT[base_of(c)]) for c in CROPS}   # heat threshold per cell (degC)
if RISK_HEAT == "mapped":                    # Tmax histograms of ERA5 (phase 2, all seasons) for the mapping
    arr, dates = read_daily("era5", ERA5_PERIOD, ("tasmax",))
    CNT_E = {c: tx_counts(arr["tasmax"], dates, CROP[c]) for c in CROPS}
    del arr
SEAS = {}                                    # (scenario, crop) -> (harvest years, {indicator: (seasons, cells)})
for s in SCEN:
    name, period = SPEC[s]
    arr, dates = read_daily(name, period)
    if s == "historical" and RISK_HEAT == "mapped":
        rows = []
        for crop in CROPS:
            thr, q = mapped_threshold(CNT_E[crop], tx_counts(arr["tasmax"], dates, CROP[crop]), HEAT_TCRIT[base_of(crop)])
            TC[crop] = thr
            w = CROP[crop]["w"]
            rows.append({"crop": crop, "Tcrit (degC)": HEAT_TCRIT[base_of(crop)],
                         "mapped threshold, area mean (degC)": np.average(thr, weights=w),
                         "shift, area p10 (degC)": float(np.percentile(thr - HEAT_TCRIT[base_of(crop)], 10)),
                         "shift, area p90 (degC)": float(np.percentile(thr - HEAT_TCRIT[base_of(crop)], 90)),
                         "quantile of Tcrit in ERA5, area mean": np.average(q, weights=w),
                         f"area beyond the ERA5 p{100 * (1 - TAIL_Q):g} (tail shift, %)": 100 * w[q > 1 - TAIL_Q].sum() / w.sum()})
        THR = pd.DataFrame(rows).set_index("crop")
        THR.to_csv(OUT / "heat_threshold_mapped.csv")
        display(THR.round(2))
    doys = dates.dayofyear.values
    ET = np.empty_like(arr["tasmax"])
    for t in range(dates.size):
        ET[t] = eto_hargreaves(arr["tasmax"][t], arr["tasmin"][t], LATR, doys[t])
    for crop in CROPS:
        SEAS[(s, crop)] = seasons(arr["tasmax"], arr["pr"], ET, dates, CROP[crop], TC[crop])
    del arr, ET
    print(f"{s}: {len(SEAS[(s, CROPS[0])][0])} seasons", flush=True)

## 2. Events relative to the reference (75th percentile of the hot days, SPEI)

SPEI: the log-logistic distribution of the water balance fitted on the reference seasons of each cell by **L-moments**,
in the generalized-logistic parametrisation of the SPEI package (Beguería et al. 2014, *Int. J. Climatol.* 34,
3001–3023; Hosking 1997, `lmom::pelglo`): k = −τ3, α = λ2 sin(kπ) / (kπ), ξ = λ1 − α (1/k − π / sin(kπ)),
F(D) = 1 / (1 + e^−y), y = −log(1 − k (D − ξ) / α) / k; SPEI = Φ^−1(F). Unlike the probability-weighted moments with
the lower bound of Vicente-Serrano et al. (2010), it also fits water balances with a long dry tail (negative skew),
common in dry climates (checked on synthetic samples: no failed fit; out of sample, with 34 reference seasons,
P(SPEI ≤ −1.5) = 7.4–8.6 % instead of 6.7 % and P(SPEI ≤ −2) = 2.3–2.9 % instead of 2.3 %). The same parameters are
applied to the runs. **Reading**: the reference is in-sample (its frequencies are the nominal ones), the runs are out of
sample and carry about +1 point from the sampling of the fit even without change: the hosing − no hosing differences
are free of it, the runs vs historical differences include it.

In [ ]:
def glo_fit(D):
    # generalized logistic fitted by sample L-moments (Hosking 1997; lmom::pelglo, as the SPEI package): D (seasons,
    # cells) of the reference -> xi, alpha, k per cell (NaN where |tau3| >= 1 or no spread)
    x = np.sort(D, axis=0)
    n = x.shape[0]
    i = np.arange(1, n + 1)[:, None]
    b0 = x.mean(0)
    b1 = (((i - 1) / (n - 1)) * x).mean(0)
    b2 = (((i - 1) * (i - 2) / ((n - 1) * (n - 2))) * x).mean(0)
    l1, l2, l3 = b0, 2 * b1 - b0, 6 * b2 - 6 * b1 + b0
    with np.errstate(invalid="ignore", divide="ignore"):
        k = -l3 / l2
        ok = (l2 > 0) & (np.abs(k) < 1)
        k = np.where(np.abs(k) < 1e-6, 1e-6, k)
        a = l2 * np.sin(k * np.pi) / (k * np.pi)
        xi = l1 - a * (1 / k - np.pi / np.sin(k * np.pi))
    return np.where(ok, xi, np.nan), np.where(ok, a, np.nan), np.where(ok, k, np.nan)


def spei(D, par):
    xi, a, k = par
    with np.errstate(invalid="ignore", divide="ignore", over="ignore"):
        arg = 1 - k * (D - xi) / a
        F = np.where(arg > 0, 1 / (1 + np.exp(np.log(np.where(arg > 0, arg, 1.0)) / k)), np.where(k > 0, 1.0, 0.0))
    return np.where(np.isfinite(xi), norm.ppf(np.clip(F, 1e-6, 1 - 1e-6)), np.nan)


EV, INFO = {}, []                            # (scenario, crop) -> {event: (seasons, cells) bool}
for crop in CROPS:
    _, ref = SEAS[("historical", crop)]
    p_hot = np.percentile(ref["HOT"], HEAT_PCT, axis=0)
    par = {k: glo_fit(ref[k]) for k in ("D_SEASON", "D_P2")}
    w = CROP[crop]["w"]
    INFO.append({"crop": crop, "cells": w.size, "reference seasons": ref["HOT"].shape[0],
                 f"area where the reference P{HEAT_PCT} of hot days is 0 (%)": 100 * w[p_hot == 0].sum() / w.sum(),
                 "area without SPEI fit (%)": 100 * w[~np.isfinite(par['D_P2'][0])].sum() / w.sum()})
    for s in SCEN:
        _, x = SEAS[(s, crop)]
        e = {"heat": x["HOT"] > p_hot[None]}
        for k, lab in (("D_SEASON", "season"), ("D_P2", "phase 2")):
            z = spei(x[k], par[k])
            for thr in SPEI_THR:
                e[f"drought {lab} (SPEI <= {thr})"] = z <= thr
        EV[(s, crop)] = e
INFO = pd.DataFrame(INFO).set_index("crop")
display(INFO.round(1))
EVENTS = list(EV[(SCEN[0], CROPS[0])])

## 3. Frequency of the events on today's cropland (area-weighted), globally and per AR6 region

In [ ]:
rows, TOP = [], {}
for crop in CROPS:
    i, j = CROP[crop]["ij"]; w = CROP[crop]["w"]
    reg = REG[i, j]
    tot = pd.Series(w).groupby(reg).sum().sort_values(ascending=False) if REG_NAME else pd.Series(dtype=float)
    TOP[crop] = [int(r) for r in tot.index[:N_REGIONS] if np.isfinite(r)]
    zones = [("global", np.ones(w.size, bool))] + [(REG_NAME[r], reg == r) for r in TOP[crop]]
    for s in SCEN:
        for ev in EVENTS:
            f = EV[(s, crop)][ev].mean(axis=0)                     # frequency of the event per cell
            for zname, z in zones:
                rows.append({"crop": crop, "zone": zname, "event": ev, "scenario": s,
                             "frequency (%)": 100 * np.average(f[z], weights=w[z])})
FREQ = pd.DataFrame(rows).set_index(["crop", "zone", "event", "scenario"])["frequency (%)"].unstack("scenario")[SCEN]
for h, o in PAIRS:
    FREQ[f"{h} - {o}"] = FREQ[h] - FREQ[o]
FREQ.to_csv(OUT / "risk_frequency.csv")
with pd.option_context("display.max_rows", 500, "display.width", 250):
    display(FREQ.xs("global", level="zone").round(1))
    display(FREQ.round(1))

## 4. Probability of exceedance of the share of today's cropland affected in a season (Ceglar et al., figure 2)

In [ ]:
COLORS = {"historical": "#2ca02c", "gcm_2025_nohosing": "#ff7f0e", "gcm_2025_hosing": "#1f77b4",
          "gcm_2050_nohosing": "#d62728", "gcm_2050_hosing": "#9467bd"}
SHOW = ["heat", f"drought phase 2 (SPEI <= {SPEI_THR[0]})"]


def exceedance(crop, ev, zone_mask):
    w = CROP[crop]["w"][zone_mask]
    out = {}
    for s in SCEN:
        share = (EV[(s, crop)][ev][:, zone_mask] * w[None]).sum(1) / w.sum()   # share of the cropland, per season
        out[s] = np.sort(share)[::-1]
    return out


for crop in CROPS:
    i, j = CROP[crop]["ij"]
    zones = [("global", np.ones(i.size, bool))] + [(REG_NAME[r], REG[i, j] == r) for r in TOP[crop]]
    fig, axes = plt.subplots(len(SHOW), len(zones), figsize=(3.2 * len(zones), 3.0 * len(SHOW)), squeeze=False, sharey=True)
    for r, ev in enumerate(SHOW):
        for k, (zname, z) in enumerate(zones):
            ax = axes[r, k]
            for s, sh in exceedance(crop, ev, z).items():
                ax.plot(sh, np.arange(1, sh.size + 1) / sh.size, color=COLORS.get(s), lw=1.5, label=s)
            ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.grid(alpha=0.3)
            ax.set_title(f"{zname}", fontsize=9)
            if k == 0:
                ax.set_ylabel(f"{ev}\nprobability of exceedance", fontsize=8)
            if r == len(SHOW) - 1:
                ax.set_xlabel("share of today's cropland", fontsize=8)
    axes[0, -1].legend(fontsize=7, loc="upper right")
    fig.suptitle(f"{crop}: seasons affected (events relative to EC-Earth3 historical {REF_PERIOD[0]}-{REF_PERIOD[1]})", fontsize=10)
    fig.tight_layout(); fig.savefig(OUT / f"exceedance_{crop}.png", dpi=90)
    plt.show() if crop == CROPS[0] else plt.close(fig)

## 5. Maps of the event frequency (maize): 2050 no hosing, 2050 hosing and the difference; hot days ERA5 vs EC-Earth3

In [ ]:
crop = "Maize_rf" if "Maize_rf" in CROPS else CROPS[0]
i, j = CROP[crop]["ij"]
pair = PAIRS[-1]


def grid_of(x):
    g = np.full(LAND.shape, np.nan)
    g[i, j] = x
    return g


fig, axes = plt.subplots(len(SHOW), 3, figsize=(30, 5.2 * len(SHOW)), squeeze=False)
for r, ev in enumerate(SHOW):
    fo, fh = (100 * EV[(s, crop)][ev].mean(axis=0) for s in (pair[1], pair[0]))
    for k, (val, title, cmap, lim) in enumerate([(fo, pair[1], "magma_r", (0, 100)), (fh, pair[0], "magma_r", (0, 100)),
                                                 (fh - fo, f"{pair[0]} - {pair[1]}", "RdBu_r", (-30, 30))]):
        ax = axes[r, k]
        ax.pcolormesh(GRID["lon"], GRID["lat"], np.where(LAND, 0, np.nan), cmap="Greys", vmin=0, vmax=4, shading="nearest")
        im = ax.pcolormesh(GRID["lon"], GRID["lat"], grid_of(val), cmap=cmap, vmin=lim[0], vmax=lim[1], shading="nearest")
        ax.set_ylim(-60, 75); ax.set_title(f"{crop}, {ev}: {title} (% of the seasons)", fontsize=10)
        plt.colorbar(im, ax=ax, shrink=0.8)
fig.savefig(OUT / f"risk_maps_{crop}.png", dpi=90, bbox_inches="tight"); plt.show()

if ERA5_CHECK:
    arr, dates = read_daily("era5", ERA5_PERIOD, ("tasmax",))
    rows = []
    for c in CROPS:
        hy, x = seasons(arr["tasmax"], np.zeros_like(arr["tasmax"]), np.zeros_like(arr["tasmax"]), dates, CROP[c], HEAT_TCRIT[base_of(c)])
        w = CROP[c]["w"]
        h_ref = SEAS[("historical", c)][1]["HOT"]
        rows.append({"crop": c, "hot days per season ERA5 (Tcrit)": np.average(x["HOT"].mean(0), weights=w),
                     f"hot days per season EC-Earth3 historical ({RISK_HEAT} threshold)": np.average(h_ref.mean(0), weights=w),
                     "area with no hot day in ERA5 (%)": 100 * w[x["HOT"].sum(0) == 0].sum() / w.sum(),
                     "area with no hot day in EC-Earth3 (%)": 100 * w[h_ref.sum(0) == 0].sum() / w.sum()})
    del arr
    CHK = pd.DataFrame(rows).set_index("crop")
    CHK.to_csv(OUT / "hot_days_era5_vs_ecearth3.csv")
    display(CHK.round(2))